# One-time setup: Snowflake catalog integration

Creates the Snowflake `CATALOG INTEGRATION` pointing at Unity Catalog's Iceberg REST
endpoint. Run this manually, once, per environment/catalog — **not** part of the
regular `f1_ingest_and_transform_workflow` job, since the integration only needs to
exist once. `sync_snowflake_iceberg_tables.ipynb` (the per-run job task) only manages
the `ICEBERG TABLE` objects against this integration; it never (re)creates it.

In [ ]:
import snowflake.connector

dbutils.widgets.text("catalog_name", "f1_analytics")
dbutils.widgets.text("secret_scope", "snowflake-iceberg-sync")

catalog_name = dbutils.widgets.get("catalog_name")
secret_scope = dbutils.widgets.get("secret_scope")

# Auto-detect the current workspace host — no need to pass it manually.
databricks_host = spark.conf.get("spark.databricks.workspaceUrl")

print(f"Catalog: {catalog_name}")
print(f"Databricks host: {databricks_host}")

In [ ]:
# All credentials come from the Databricks secret scope Terraform populates.
# Nothing sensitive is ever hard-coded here or passed as a plain job parameter.
snowflake_account = dbutils.secrets.get(secret_scope, "snowflake-account")
snowflake_user = dbutils.secrets.get(secret_scope, "snowflake-service-user")
snowflake_private_key_pem = dbutils.secrets.get(secret_scope, "snowflake-service-user-private-key")
snowflake_role = dbutils.secrets.get(secret_scope, "snowflake-role")
snowflake_warehouse = dbutils.secrets.get(secret_scope, "snowflake-warehouse")
snowflake_database = dbutils.secrets.get(secret_scope, "snowflake-database")
snowflake_target_schema = dbutils.secrets.get(secret_scope, "snowflake-schema")

databricks_sp_client_id = dbutils.secrets.get(secret_scope, "databricks-sp-client-id")
databricks_sp_client_secret = dbutils.secrets.get(secret_scope, "databricks-sp-client-secret")

In [ ]:
from cryptography.hazmat.primitives import serialization

# snowflake-connector-python wants the key as DER bytes, not the PEM string stored in the secret scope.
private_key_der = serialization.load_pem_private_key(
    snowflake_private_key_pem.encode(), password=None
).private_bytes(
    encoding=serialization.Encoding.DER,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption(),
)

conn = snowflake.connector.connect(
    account=snowflake_account,
    user=snowflake_user,
    private_key=private_key_der,
    role=snowflake_role,
    warehouse=snowflake_warehouse,
    database=snowflake_database,
    schema=snowflake_target_schema,
)
cur = conn.cursor()

In [ ]:
# Must match the formula sync_snowflake_iceberg_tables.ipynb uses to reference this
# integration by name — both derive it from catalog_name alone, so nothing needs to
# be passed between the two notebooks.
catalog_integration_name = f"uc_{catalog_name}_iceberg"
catalog_uri = f"https://{databricks_host}/api/2.1/unity-catalog/iceberg-rest"
oauth_token_uri = f"https://{databricks_host}/oidc/v1/token"

# Syntax confirmed against Snowflake's "Configure a catalog integration for Unity
# Catalog" doc page — note CATALOG_SOURCE = ICEBERG_REST (one token, underscore, not
# two words), ACCESS_DELEGATION_MODE nested inside REST_CONFIG (not top-level), and
# OAUTH_TOKEN_URI/OAUTH_ALLOWED_SCOPES required for Databricks specifically (Databricks
# is not the OAuth token issuer by default, so it must be pointed at explicitly). No
# CATALOG_NAMESPACE here — this integration has no default namespace, so
# sync_snowflake_iceberg_tables.ipynb sets CATALOG_NAMESPACE per table instead. Still a
# Public Preview feature — re-verify against current docs if this starts failing again.
create_integration_sql = f"""
CREATE CATALOG INTEGRATION IF NOT EXISTS {catalog_integration_name}
CATALOG_SOURCE = ICEBERG_REST
TABLE_FORMAT = ICEBERG
REST_CONFIG = (
  CATALOG_URI = '{catalog_uri}'
  CATALOG_NAME = '{catalog_name}'
  ACCESS_DELEGATION_MODE = VENDED_CREDENTIALS
)
REST_AUTHENTICATION = (
  TYPE = OAUTH
  OAUTH_TOKEN_URI = '{oauth_token_uri}'
  OAUTH_CLIENT_ID = '{databricks_sp_client_id}'
  OAUTH_CLIENT_SECRET = '{databricks_sp_client_secret}'
  OAUTH_ALLOWED_SCOPES = ('all-apis')
)
ENABLED = TRUE
"""
cur.execute(create_integration_sql)
print(f"Catalog integration '{catalog_integration_name}' ready")

In [ ]:
# Confirms the integration can actually reach Unity Catalog's REST endpoint and
# authenticate — separate from whether the CREATE statement itself parsed correctly.
cur.execute(f"SELECT SYSTEM$VERIFY_CATALOG_INTEGRATION('{catalog_integration_name}')")
print(cur.fetchone()[0])

In [ ]:
cur.close()
conn.close()